In [ ]:
import ee, geemap, os, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try: ee.Initialize(project='geeexercise')
except: ee.Authenticate(); ee.Initialize(project='geeexercise')

In [ ]:
# Configurations
aoi = ee.FeatureCollection('projects/geeexercise/assets/gebe-island-aoi-land').geometry()
yrs = range(2019, 2026)
crs, scale = 'EPSG:32752', 10

feat_bands = ['NDVI','EVI','MSAVI2','BSI','NDWI','NBR','Clay','Ferrous','VV','VH','VV_VH_ratio']
all_12     = [f'b{i}' for i in range(1,13)]   # GEE strips band names on ingest

# LULC schema
codes = [1,2,3,4,5]
lbls  = ['vegetation','baresoil','bush','builtup','waterbodies']
pal   = ['#1a9641','#d2b48c','#a6d96a','#d7191c','#2b83ba']
vis   = {'min':1, 'max':5, 'palette':pal}

# Asset paths
stack_tmpl = 'projects/geeexercise/assets/new-thesis-img/FeatureStack_{yr}'
samp_base  = 'projects/geeexercise/assets/thesis'
samp_nm    = 'new_sample'
code_fld, set_fld = 'code', 'set_type'

# Export destinations
rast_dir = 'gee-exports-new/phase-3-new'
csv_dir  = 'gee-exports-new/phase-3-new/csv'
os.makedirs(csv_dir, exist_ok=True)

In [ ]:
# Load feature stacks
all_12 = ['NDVI','EVI','MSAVI2','BSI','NDWI','NBR','Clay','Ferrous','IronOxide','VV','VH','VV_VH_ratio']

def pull_stack(yr):
    return (ee.Image(stack_tmpl.format(yr=yr))
              .rename(all_12)
              .select(feat_bands)
              .clip(aoi))

stacks = {yr: pull_stack(yr) for yr in yrs}

# Quick sanity check
sample_yr = 2022
bands_ok = stacks[sample_yr].bandNames().getInfo() == feat_bands
print(f"Loaded {len(stacks)} stacks, band names {'OK' if bands_ok else 'MISMATCH'}")

In [ ]:
# Load pre-split samples (no random re-split, set_type is fixed from collection's attribute table)
trn_fc, tst_fc = {}, {}
for yr in yrs:
    fc = ee.FeatureCollection(f'{samp_base}/{samp_nm}_{yr}')
    trn_fc[yr] = fc.filter(ee.Filter.eq(set_fld, 'training'))
    tst_fc[yr] = fc.filter(ee.Filter.eq(set_fld, 'testing'))

# Pull total counts once to sanity-check the split. In this case: 2022
ref = 2022
n_tr, n_te = trn_fc[ref].size().getInfo(), tst_fc[ref].size().getInfo()
print(f"[{ref}] train={n_tr}  test={n_te}  ratio={n_tr/(n_tr+n_te):.0%}/{n_te/(n_tr+n_te):.0%}")

In [ ]:
# Extract pixel values, but triggers on train()
trn = {yr: stacks[yr].sampleRegions(trn_fc[yr], [code_fld], scale, tileScale=8, geometries=False) for yr in yrs}
tst = {yr: stacks[yr].sampleRegions(tst_fc[yr], [code_fld], scale, tileScale=8, geometries=False) for yr in yrs}
print('Sample regions defined (lazy).')

In [ ]:
# RF hyperparameters tuning
n_trees, bag_frac, leaf, seed = 200, 0.5, 2, 42

clfs, lulc_maps, oob, imp_raw = {}, {}, {}, {}

print('Training RF classifiers...')
for yr in yrs:
    clf = (ee.Classifier.smileRandomForest(
               numberOfTrees=n_trees,
               bagFraction=bag_frac,
               minLeafPopulation=leaf,
               seed=seed)
             .setOutputMode('CLASSIFICATION')
             .train(trn[yr], code_fld, feat_bands))

    expl = clf.explain().getInfo()
    oob[yr]     = expl.get('outOfBagErrorEstimate')
    imp_raw[yr] = expl.get('importance', {})

    lulc_maps[yr] = (stacks[yr].classify(clf)
                                .rename('classification')
                                .toUint8()
                                .clip(aoi)
                                .set('year', yr))
    clfs[yr] = clf

    oob_str = f"{oob[yr]:.4f}" if oob[yr] else 'N/A'
    print(f'  {yr}: OOB={oob_str}')

print(f'Done. {len(clfs)} models trained.')


In [ ]:
# Accuracy assessment on held-out test samples
acc_res, conf_mats, acc_dfs, conf_dfs = {}, {}, {}, {}

print('Computing accuracy metrics...')
for yr in yrs:
    pred_field = 'classification'
    test_pred = tst[yr].classify(clfs[yr], pred_field)
    em = test_pred.errorMatrix(code_fld, pred_field, codes).getInfo()
    conf_mats[yr] = em

    n  = sum(sum(r) for r in em)
    tp = sum(em[i][i] for i in range(len(codes)))
    oa = tp / n if n else 0

    row_s = [sum(r) for r in em]
    col_s = [sum(em[r][c] for r in range(len(codes))) for c in range(len(codes))]
    pe    = sum(r*c for r,c in zip(row_s, col_s)) / n**2
    kappa = (oa - pe) / (1 - pe) if (1 - pe) else 0

    pa, ua, f1 = {}, {}, {}
    for i, nm in enumerate(lbls):
        p = em[i][i] / row_s[i] if row_s[i] else 0
        u = em[i][i] / col_s[i] if col_s[i] else 0
        pa[nm] = round(p*100, 2)
        ua[nm] = round(u*100, 2)
        f1[nm] = round(2*p*u/(p+u)*100, 2) if (p+u) else 0

    acc_res[yr] = {'OA': round(oa*100,2), 'Kappa': round(kappa,4),
                   'OOB': round(oob[yr],4) if oob[yr] else None,
                   'PA': pa, 'UA': ua, 'F1': f1}

    # Per-class DataFrame + confusion matrix DataFrame
    acc_dfs[yr]  = pd.DataFrame([{'Class':nm,'PA(%)':pa[nm],'UA(%)':ua[nm],'F1(%)':f1[nm]} for nm in lbls])
    conf_dfs[yr] = pd.DataFrame(em, index=[f'Act:{l}' for l in lbls], columns=[f'Pred:{l}' for l in lbls])

    print(f'  {yr}: OA={acc_res[yr]["OA"]}%  Kappa={acc_res[yr]["Kappa"]}  OOB={oob[yr]:.4f}')


In [ ]:
# Cross-year summary
rows = []
for yr in yrs:
    m = acc_res[yr]
    rows.append({'Year':yr, 'OA(%)':m['OA'], 'Kappa':m['Kappa'], 'OOB':m['OOB'],
                 **{f'PA_{l[:4]}':m['PA'][l] for l in lbls},
                 **{f'F1_{l[:4]}':m['F1'][l] for l in lbls}})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
print(f"\nMean OA={summary_df['OA(%)'].mean():.2f}%  Mean Kappa={summary_df['Kappa'].mean():.4f}")

In [ ]:
# Feature importance
imp_dfs, rows = {}, []
for yr in yrs:
    raw = imp_raw.get(yr, {})
    tot = sum(raw.values()) or 1
    normed = {b: raw.get(b,0)/tot for b in feat_bands}
    df = (pd.DataFrame([{'Feature':b,'Importance':round(normed[b],6)} for b in feat_bands])
            .sort_values('Importance', ascending=False).reset_index(drop=True))
    df.index += 1; df.index.name = 'Rank'
    imp_dfs[yr] = df
    for _, r in df.iterrows():
        rows.append({'Year':yr,'Feature':r['Feature'],'Importance':r['Importance']})

all_imp_df = pd.DataFrame(rows)
avg_imp = (all_imp_df.groupby('Feature')['Importance'].mean()
                     .reset_index().rename(columns={'Importance':'Avg'})
                     .sort_values('Avg', ascending=False).reset_index(drop=True))
avg_imp.index += 1; avg_imp.index.name = 'Rank'

print('Average feature importance (2019-2025):')
print(avg_imp.to_string())


In [ ]:
# Plot importance - per-band + category rollup
spec = feat_bands[:6]; geo = feat_bands[6:8]; sar = feat_bands[9:]
cat_clr = {'Spectral':'#2a7d3f','Geologic':'#b85c1a','SAR':'#1a4a8a'}
def band_clr(b):
    if b in spec: return cat_clr['Spectral']
    if b in geo:  return cat_clr['Geologic']
    return cat_clr['SAR']

plot_s = avg_imp.set_index('Feature')['Avg'].sort_values()

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(13,5))

bars = ax1.barh(plot_s.index, plot_s.values,
                color=[band_clr(b) for b in plot_s.index], height=0.7)
ax1.set_xlabel('Mean Normalised Gini Importance')
ax1.set_title('Feature Importance - Band Level', fontweight='bold')
ax1.axvline(1/len(feat_bands), color='grey', ls='--', lw=0.8, label=f'Equal ({1/len(feat_bands):.3f})')
for bar in bars:
    ax1.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
             f'{bar.get_width():.3f}', va='center', fontsize=8)
ax1.legend(handles=[mpatches.Patch(color=v,label=k) for k,v in cat_clr.items()], fontsize=8)

imp_lkp = avg_imp.set_index('Feature')['Avg']
cat_vals = {
    'Spectral\n(6)': sum(imp_lkp.get(b,0) for b in spec),
    'Geologic\n(3)': sum(imp_lkp.get(b,0) for b in geo),
    'SAR\n(3)':      sum(imp_lkp.get(b,0) for b in sar),
}
cbars = ax2.bar(cat_vals.keys(), cat_vals.values(), color=list(cat_clr.values()), width=0.5)
ax2.set_ylabel('Sum of Mean Importance')
ax2.set_title('Category Rollup', fontweight='bold')
for bar in cbars:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{bar.get_height():.3f}', ha='center', fontweight='bold')
ax2.set_ylim(0, max(cat_vals.values())*1.2)

plt.suptitle('RF Gini Feature Importance - Gebe Island 2019-2025', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{csv_dir}/feat_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# OOB error trend + test error overlay
fig, ax = plt.subplots(figsize=(8,4))
oob_vals = [oob[yr] for yr in yrs]
err_vals  = [1 - acc_res[yr]['OA']/100 for yr in yrs]

ax.plot(list(yrs), oob_vals,  'o-', color='#c0392b', lw=2, ms=7, label='OOB Error (train)')
ax.plot(list(yrs), err_vals,  's--', color='#2980b9', lw=2, ms=7, label='Test Error (1-OA)')
for yr, v in zip(yrs, oob_vals):
    ax.annotate(f'{v:.3f}', (yr, v), textcoords='offset points', xytext=(0,8),
                ha='center', fontsize=8, color='#c0392b')

ax.axhline(0.05, color='grey', ls=':', lw=1, label='5% reference')
ax.set_xticks(list(yrs)); ax.set_xlabel('Year'); ax.set_ylabel('Error Rate')
ax.set_title(f'OOB Error Trend - RF({n_trees} trees) Gebe Island', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'{csv_dir}/oob_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# QA visualization
lat, lon = aoi.centroid().coordinates().getInfo()[::-1]

Map = geemap.Map(center=[lat, lon], zoom=11)
Map.add_basemap('SATELLITE')

# AOI boundary
Map.addLayer(ee.FeatureCollection([ee.Feature(aoi)]).style(
    **{'color':'#FFFFFF','fillColor':'00000000','width':2}), {}, 'AOI')

for yr in yrs:
    Map.addLayer(lulc_maps[yr], vis, f'LULC {yr}', shown=(yr==2025))

legend = {f'{n.capitalize()} ({c})': p for n,c,p in zip(lbls, codes, pal)}
Map.add_legend(title='LULC', legend_dict=legend, position='bottomright')
Map.addLayerControl()
Map

In [ ]:
# Submit raster exports
tasks = []
print(f'Submitting {len(list(yrs))} raster exports to Drive:{rast_dir}...')
for yr in yrs:
    fname = f'BARU_LULC_Gebe_{yr}_RF{n_trees}trees' # define your own name 😂. "BARU" means "NEW"
    t = ee.batch.Export.image.toDrive(
        image=lulc_maps[yr], description=fname,
        folder=rast_dir, fileNamePrefix=fname,
        region=aoi, scale=scale, crs=crs,
        maxPixels=1e13, fileFormat='GeoTIFF')
    t.start(); tasks.append({'yr':yr,'name':fname,'task':t})
    print(f'  STARTED: {fname}.tif')
print('Done. Run next cell to monitor.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

csv_dir  = '/content/drive/MyDrive/gee-exports-new/phase-3/csv'
rast_dir = 'gee-exports-new/phase-3'
os.makedirs(csv_dir, exist_ok=True)

In [ ]:
# Write CSVs
print(f'Writing CSVs to {csv_dir}...')
for yr in yrs:
    acc_dfs[yr].assign(Year=yr).to_csv(f'{csv_dir}/acc_{yr}.csv', index=False)
    conf_dfs[yr].to_csv(f'{csv_dir}/conf_{yr}.csv')
    imp_dfs[yr].drop(columns=[], errors='ignore').assign(Year=yr).to_csv(f'{csv_dir}/imp_{yr}.csv')

summary_df.to_csv(f'{csv_dir}/acc_summary.csv', index=False)
avg_imp.to_csv(f'{csv_dir}/imp_avg.csv')
all_imp_df.to_csv(f'{csv_dir}/imp_all_years.csv', index=False)

print(f'CSVs written: per-year acc/conf/imp + summary + avg')

In [ ]:
# Re-run to refresh export status
print(f'Export status ({len(tasks)} raster tasks):')
print(f'{"#":>3}  {"File":<38} {"State"}')
done = 0
for i, t in enumerate(tasks, 1):
    s = t['task'].status()['state']
    if s == 'COMPLETED': done += 1
    print(f'{i:>3}  {t["name"]:<38} {s}')
print(f'\n{done}/{len(tasks)} completed. Re-run to refresh.')